# Opening NISAR GCOV HDF5 Files in Python

<br>

This notebook demonstrates how to open and access NISAR GCOV HDF5 files using `h5py` (Python), focusing on navigating GCOV files to locate and extract data of interest.

This notebook is intended for users with basic Python experience who want to work directly with NISAR GCOV products, including scientists, students, and practitioners transitioning from GeoTIFF-based SAR products to HDF5-based formats.

<hr>

## Overview

1. [Prerequisites](hdf5-prereqs)
1. [Download a NISAR GCOV HDF5 file](hdf5-download)
1. [Accessing the science group](hdf5-science)
1. [Accessing the LSAR group](hdf5-lsar)
1. [Accessing the GCOV group](hdf5-gcov)
1. [Navigating into the grids group](hdf5-grids)
1. [Inspecting datasets within a frequency group](hdf5-frequency)
1. [Accessing a specific GCOV dataset](hdf5-gcov_dataset)
1. [Reading the dataset into memory](hdf5-readmemory)
1. [Build a raster transform and export to GeoTIFF](hdf5-geotiff)
1. [Summary](hdf5-summary)
1. [Resources and References](hdf5-resources)

<hr>

(hdf5-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/software_environment_docker.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

(hdf5-download)=
## 2. Download a NISAR GCOV HDF5 file

Download a GCOV HDF5 file to your home directory and view the top-level HDF5 group, `science`.

### 2a. Download the HDF5 file

In [1]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass
from pathlib import Path

#1: Search for and download NISAR GCOV HDF5 products from ASF

#authenticate with Earthdata Login
#create an ASF session object (stores your login + settings for requests)
session = asf.ASFSession()
#prompt for username/password
session.auth_with_creds(input('EDL Username'), getpass('EDL Password'))

#2: Define the search window and area of interest 

#insert start and end dates. Note that dates and times are inclusive. 
start_date = datetime(2025, 12, 4)
end_date = datetime(2025, 12, 5)
# POINT or POLYGON as WKT (well-known-text) using lat/lon pairs 
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" 

#3: Build search options 

#these filters narrow results to NISAR GCOV products over the AOI and time window
opts = asf.ASFSearchOptions(
    **{
        "maxResults": 250,                    # cap the number of returned results
        "intersectsWith": area_of_interest,   # spatial filter (WKT geometry)
        "start": start_date,                  # temporal filter (start)
        "end": end_date,                      # temporal filter (end)
        "processingLevel": ["GCOV"],          # product type / processing level
        "dataset": ["NISAR"],                 # mission/dataset name
        "productionConfiguration": ["PR"],    # production config (e.g., provisional)
        "session": session,                   # use authenticated session from above
    }
)

#4: Run the search for HDF5 files
#execute search
response = asf.search(opts=opts)

#exclude QA_STATE files 
pattern = r'^(?!.*QA_STATS).*'

#extract files that end with .h5 
hdf5_files = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
hdf5_files

EDL Username jwhite124
EDL Password ········


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5']

### 2b. Retain only the URL for the most recent version of each product in the search results

Data is occasionally re-released with an updated version. Versions are recorded as a [Composite Release Identifier (CRID)](https://nisar-docs.asf.alaska.edu/naming-conventions/) in a product's filename. We can use the CRID to retain only the most recent version of each product in the list of URLs.

In [2]:
import re

pattern = re.compile(r"(NISAR_L2_PR_GCOV(?:_[^_]+){9})_(X\d{5})")

latest_version_dict = {}

for url in hdf5_files:
    m = pattern.search(url)
    if not m:
        continue

    product, crid = m.groups()

    if product not in latest_version_dict or crid > latest_version_dict[product][0]:
        latest_version_dict[product] = (crid, url)

hdf5_files = [i[1] for i in latest_version_dict.values()]
print(f"Retained {len(hdf5_files)} GCOV products:")
hdf5_files

Retained 1 GCOV products:


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5']

### 2c. Download the data

In [3]:
# Create a folder and download the files
data_dir = Path.home() / "GCOV_data"
data_dir.mkdir(exist_ok=True)
print(f'data_dir: {data_dir}')

#download files
asf.download_urls(hdf5_files, data_dir, session=session, processes=4)

data_dir: /home/jovyan/GCOV_data


### 2d. Open the file and list the top-level HDF5 group

In [4]:
import h5py

#path to the NISAR GCOV HDF5 file
h5_path = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

#open the GCOV HDF5 file in read mode ("r")
#HDF5 files behave like Python dictionaries, where keys correspond to groups
f = h5py.File(h5_path, "r")

#inspect the top-level group names in the file
list(f)

['science']

<hr>

(hdf5-science)=
## 3. Accessing the science and LSAR groups

All NISAR science products organize their primary data and metadata under the `/science` group. After opening the file, the first step in navigating a GCOV product is to inspect this group to determine which SAR instruments (LSAR only, or both SSAR and LSAR) are included in the file. This indicates available paths within the GCOV data.

In this notebook, we focus exclusively on GCOV data produced by the L-band SAR (LSAR) instrument.

In [5]:
list(f["science"])

['LSAR']

:::{dropdown} Output Explanation 

The output of this command shows the available science subgroups within the file. Depending on the product and acquisition mode, a GCOV file may contain:

- only `LSAR`: L-band SAR data, or

- both `LSAR` and `SSAR`: L-band and S-band SAR data

`L-SAR` and `S-SAR` data are always stored in separate groups, even if they cover the same geographic area or acquisition time.
:::

<hr>

(hdf5-lsar)=
## 4. Accessing the LSAR group

Within the `/science/LSAR`/ group, different product types and supporting information are organized into separate subgroups. For GCOV products, all gridded covariance data are stored under the `/science/LSAR/GCOV/` path.

The next step is to inspect the contents of the `/science/LSAR/` group to confirm the presence of the GCOV subgroup.

In [6]:
#list the contents of the /science/LSAR group
#this shows the available product types and supporting subgroups
list(f["science/LSAR"])

['GCOV', 'identification']

::: {dropdown} Output Explanation 
The `/science/LSAR/` group contains two subgroups:

- `GCOV`: the gridded covariance product data

- `identification`: granule-level metadata describing the acquisition and processing of this product

The presence of the GCOV group confirms that this file contains GCOV data, and indicates where the gridded raster datasets are stored. The identification group contains supporting metadata and is useful for context, but note that it does not contain raster data.
:::

<hr>

(hdf5-gcov)=
## 5. Accessing the GCOV group

After confirming that the file contains GCOV data, the next step is to examine how the GCOV product is organized internally. The `GCOV` group serves as the top-level container for all gridded covariance data and related metadata within the LSAR science product.

Listing the contents of the `GCOV` group allows us to identify where the gridded measurement datasets are stored and which subgroups contain supporting information.

In [7]:
#list the contents of the GCOV group
#this reveals how GCOV data are organized internally
list(f["science/LSAR/GCOV"])

['grids', 'metadata']

::: {dropdown} Output Explanation 
The GCOV group is organized into two primary subgroups:

- `grids`: contains the gridded covariance datasets and related raster-style data

- `metadata`: contains product-level metadata specific to the GCOV product

All GCOV raster data that is commonly visualized or analyzed are stored under the grids group. The metadata group provides supporting information but does not contain gridded measurement arrays.
:::

<hr>

(hdf5-grids)=
## 6. Navigating into the grids group
The contents of the `grids` group inform which frequency groups are present in the GCOV product. Depending on the acquisition mode, a GCOV file may contain data from one or two frequency groups.

In [8]:
#list the contents of the grids group
#this shows how GCOV raster data are organized by frequency
list(f["science/LSAR/GCOV/grids"])

['frequencyA', 'frequencyB']

::: {dropdown} Output Explanation 
GCOV raster datasets are grouped by frequency band within the `grids` group. The two frequency groups represent different portions of the radar bandwidth:

- `frequencyA`: datasets derived from the lower portion of the LSAR bandwidth

- `frequencyB`: datasets derived from the higher portion of the LSAR bandwidth

Depending on the acquisition mode, a GCOV product may contain data from only one frequency group or from both `frequencyA` and `frequencyB`. The presence of both groups indicates that this product includes measurements from two portions of the L-SAR spectrum.
:::

<hr>

(hdf5-frequency)=
## 7. Inspecting datasets within a frequency group

The next step is to list the datasets available for a specific frequency group. In this example, we inspect `frequencyA`.

In [9]:
#list the datasets available within the frequencyA group
#these correspond to specific polarization combinations and supporting layers
list(f["science/LSAR/GCOV/grids/frequencyA"])

['HHHH',
 'HVHV',
 'listOfCovarianceTerms',
 'listOfPolarizations',
 'mask',
 'numberOfLooks',
 'numberOfSubSwaths',
 'projection',
 'rtcGammaToSigmaFactor',
 'xCoordinateSpacing',
 'xCoordinates',
 'yCoordinateSpacing',
 'yCoordinates']

::: {dropdown} Output Explanation 
#### 1. Gridded covariance (measurement) datasets

These datasets contain the primary GCOV raster data and represent specific polarization combinations. Examples include:

- `HHHH`, `HVHV`: real-valued covariance terms

Other products may also include cross-polarized or complex-valued terms. These datasets are commonly extracted, visualized, or converted to raster formats.

#### 2. Supporting raster layers

These datasets provide information needed to interpret or process radar measurement data:

- `numberOfLooks`: effective number of looks for each pixel

- `rtcGammaToSigmaFactor`: factor used to convert radiometrically terrain-corrected gamma values to sigma-naught

- `mask`: identifies valid and invalid pixels

These layers are spatially aligned with the covariance datasets.

#### 3. Grid and coordinate information

These datasets define the spatial reference of the GCOV grids:

- `xCoordinates`, `yCoordinates`: coordinate vectors defining the grid

- `xCoordinateSpacing`, `yCoordinateSpacing`: grid spacing

- `projection`: map projection information
:::

<hr>

(hdf5-gcov_dataset)=
## 8. Accessing a specific GCOV dataset

Once the available datasets within a frequency group have been identified, the next step is to access an individual dataset directly. Before loading the full raster into memory, it is useful to inspect basic properties of the dataset, such as its corresponding shape and data type, to confirm that it contains the expected data.

In this example, we access the `HVHV` dataset from the `frequencyA` group.

In [10]:
#access a covariance (measurement) dataset by its full HDF5 path
#HVHV is a real-valued covariance term for the HV–HV polarization combination
hvhv = f["science/LSAR/GCOV/grids/frequencyA/HVHV"]

#inspect basic properties of the dataset without loading the full array
hvhv.shape, hvhv.dtype

((16704, 17064), dtype('<f4'))

::: {dropdown} Output Explanation

- The `HVHV` object is an HDF5 dataset representing a gridded covariance measurement.

- The shape (16704, 17064) indicates the number of rows (16704) and columns (17064) in the raster grid.

- The data type ('<f4') describes how the values are stored (in this example, 32-bit floating point).

Accessing these properties does not load the full dataset into memory, making this a safe way to verify dataset contents before reading the data values.

### 8a. Inspecting dataset attributes

Each GCOV dataset includes a set of attributes that describe how the data values should be interpreted. These attributes provide essential context such as units, valid value ranges, and how missing or invalid data are represented.

Before reading the dataset into memory or using it in analysis, we inspect these attributes to understand what the values might represent.

In [11]:
#list all attributes attached to the HVHV dataset
list(hvhv.attrs)

['DIMENSION_LIST',
 '_FillValue',
 'description',
 'grid_mapping',
 'long_name',
 'max_value',
 'mean_value',
 'min_value',
 'sample_stddev',
 'units',
 'valid_min']

In [12]:
#select key dataset attributes
#before reading the dataset into memory, we inspect a small number of attributes that directly affect how the data should be handled in analysis and visualization

print("Fill value:", hvhv.attrs.get("_FillValue"))
print("Minimum valid value:", hvhv.attrs.get("valid_min"))

Fill value: nan
Minimum valid value: 0.0


::: {dropdown} Output Explanation

- The` _FillValue` attribute indicates how missing or invalid pixels are represented. In this dataset, missing values are stored as `NaN`, which are automatically handled by NumPy and plotting libraries.

- The `valid_min` attribute indicates that valid values are expected to be non-negative. 
In this dataset, valid values are expected to be non-negative, which provides a simple sanity check when plotting or analyzing the data.

<hr>

(hdf5-readmemory)=
## 9. Reading the dataset into memory

Up to this point, we have been working with an HDF5 dataset object, which provides access to metadata without loading the data values themselves. To perform calculations, visualization, or raster export, the dataset values must be explicitly read into memory.

In this step, we load the `HVHV` dataset into a NumPy array.

In [13]:
#read the full HVHV dataset into memory as a NumPy array
hvhv_data = hvhv[()]

#confirm the array shape
hvhv_data.shape

(16704, 17064)

::: {dropdown} Output Explanation

- hvhv_data is now a NumPy array containing the gridded covariance values.

- The array shape matches the dimensions reported earlier by hvhv.shape.

- Missing values are represented as NaN, consistent with the _FillValue attribute.

At this point, the GCOV raster values are available for direct analysis, visualization, or conversion to a standard raster format.

### 9a. Reading the grid coordinate vectors

GCOV datasets are stored as gridded arrays, and the file provides coordinate datasets that define the spatial location of each pixel. The `xCoordinates` and `yCoordinates` datasets provide the coordinate vectors for the grid, which are used to geolocate the raster.

In [14]:
#read the x and y coordinate vectors for the frequencyA grid
x = f["science/LSAR/GCOV/grids/frequencyA/xCoordinates"][:]
y = f["science/LSAR/GCOV/grids/frequencyA/yCoordinates"][:]

#inspect coordinate ranges and lengths
print(f"x range: {x.min():.1f} to {x.max():.1f}")
print(f"y range: {y.min():.1f} to {y.max():.1f}")
print(f"number of x coordinates (columns): {len(x)}")
print(f"number of y coordinates (rows): {len(y)}")

x range: 604090.0 to 945350.0
y range: 1254250.0 to 1588310.0
number of x coordinates (columns): 17064
number of y coordinates (rows): 16704


::: {dropdown} Output Explanation
- `x` and `y` are one-dimensional coordinate vectors that define the grid in the product’s projected coordinate system.

- The length of `x` corresponds to the number of raster columns, and the length of `y` corresponds to the number of raster rows. These should match the dimensions of `hvhv_data`.

- The minimum and maximum values of `x` and `y` define the spatial extent of the GCOV grid.
:::

### 9b. Reading grid spacing

The `xCoordinates` and `yCoordinates` vectors define the grid location, but we also need the spacing between pixels. GCOV products provide `xCoordinateSpacing` and `yCoordinateSpacing`, which describe the pixel size in the projected coordinate system. These values are useful for confirming resolution and for constructing a georeferencing transform during raster export.

In [15]:
#read the grid spacing (pixel size) for the frequencyA grid
dx = f["science/LSAR/GCOV/grids/frequencyA/xCoordinateSpacing"][()]
dy = f["science/LSAR/GCOV/grids/frequencyA/yCoordinateSpacing"][()]

#print
print(f"x spacing (dx): {float(dx)}")
print(f"y spacing (dy): {float(dy)}")

x spacing (dx): 20.0
y spacing (dy): -20.0


::: {dropdown} Output Explanation

- The grid spacing in the x-direction (`dx`) is 20 meters, indicating a pixel width of 20 m in the projected coordinate system.

- The grid spacing in the y-direction (`dy`) is −20 meters. The negative sign indicates that y-coordinates decrease from north to south, which is a common convention for raster grids.

- Together, these values indicate that the GCOV data are provided on a 20 m × 20 m grid.

This spacing information, combined with the coordinate vectors, is used to construct a georeferencing transform when converting the GCOV dataset to a standard raster format.
:::

### 9c. Reading projection information

The GCOV grid is defined in a projected coordinate system. To correctly interpret the coordinate values and export a georeferenced raster, we need to identify the projection used by the grid.

GCOV products store projection information within each frequency group.

In [16]:
#read the projection information for the frequencyA grid
projection = f["science/LSAR/GCOV/grids/frequencyA/projection"][()]

projection

np.uint32(32637)

::: {dropdown} Output Explanation

- The projection value is 32637, stored as an unsigned integer (`np.uint32`).

- This is an EPSG code, meaning the GCOV grid uses the coordinate reference system EPSG:32637.

- The `xCoordinates` and `yCoordinates` values are therefore in meters in this projected CRS.

With the CRS identified, we can now create a georeferencing transform and export the `HVHV` dataset to a GeoTIFF with a valid CRS.
:::

<hr>

(hdf5-geotiff)=
## 10. Build a raster transform and export to GeoTIFF

Now we combine:
- coordinate vectors (`x`, `y`)
- pixel spacing (`dx`, `dy`)
- CRS (EPSG:32637) 

to export a properly georeferenced raster.

In [17]:
import numpy as np
import rasterio
from rasterio.transform import from_origin

#convert the projection value into a CRS string for rasterio 
crs = f"EPSG:{int(projection)}"

#NISAR x/y coordinates represent pixel centers
#Rasterio needs the upper-left pixel corner, so a half pixel shift is required
# transform = from_origin(
#     x.min() - float(dx) / 2,
#     y.max() + float(abs(dy)) / 2,
#     float(dx),
#     float(abs(dy))
# )
start_x = float(x[0]) - float(dx) / 2
start_y = float(y[0]) - float(dy) / 2

transform = from_origin(
    start_x,
    start_y,
    float(dx),
    float(abs(dy))
)

#name of the output GeoTIFF file
out_tif = data_dir / "GCOV_frequencyA_HVHV.tif"

#open a new GeoTIFF file 
with rasterio.open(
    out_tif,
    "w",                      #write mode
    driver="GTiff",           #output file format
    height=hvhv_data.shape[0],#number of rows
    width=hvhv_data.shape[1], #number of columns
    count=1,                  #single-band raster
    dtype=hvhv_data.dtype,    #data type matches the NumPy array
    crs=crs,                  #coordinate reference system
    transform=transform,      #georeferencing information
    nodata=np.nan,            #missing values stored as NaNs
) as dst:
    #write the HVHV data array to band 1 of the GeoTIFF
    dst.write(hvhv_data, 1)

#print the output filename for confirmation of file creation
out_tif

PosixPath('/home/jovyan/GCOV_data/GCOV_frequencyA_HVHV.tif')

In [18]:
#quality check
with rasterio.open(out_tif) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)


CRS: EPSG:32637
Resolution: (20.0, 20.0)
Bounds: BoundingBox(left=604080.0, bottom=1254240.0, right=945360.0, top=1588320.0)


<hr>

(HDF5-summary)=
## 11. Summary
You have now opened and navigated an HDF5 file, saved it to memory, and exported a layer as a GeoTIFF. 
<hr>

(hdf5-resources)=
## 12. Resources and references
- [GCOV Specifications](https://d1mv8zhcvry6x4.cloudfront.net/s3-7fdfbcb0ce308dc58d08f97acb1f0590/sds-n-cumulus-prod-nisar-sample-data.s3.us-west-2.amazonaws.com/DOCS/NISAR_D-102274_RevE_NASA_SDS_Product_Specification_L2_GCOV_Nov8_2024_w-sigs.pdf?A-userid=None&Expires=1772497215&Signature=FTbel0AsOFTbJljWTPumbhEkBWUgv3DHAeO4cQ7lYx-nwsgUycC3YDjIHnT--sFwvrnG9I8g2-~InboeRqWavjXbM1pfl-l1Ek9JNqMm0tkFn9Zpraa3oGdRX5671OFfspQF2eH04XYzBAiFCuun1CnAYoYrfQ5Y9u~7Y4IxqvV5wX8vZA3fZBDqX3jh7pibEe-2ZAAS9a-O0K~IsxGEV7F9xzu3Fd3VYe4~emOsF79b8kv1nPlPLFPrOXgNX5ftggAu1jUna2A63DI7CWRkTIXo4GvAebhMeubss66iSgpPortVQBr9zt2qGJ6xk~ygfMEt5Vgk4Dl3NQs1Aocwzw__&Key-Pair-Id=K1MF06CKIT7LR7)
- [HDF5 Documentation](https://support.hdfgroup.org/documentation/)
  
**Authors:** Julia White, Alex Lewandowski

<hr>